# 01 — Data Exploration

**Goal:** Understand the product dataset before building the similarity search index.

Covers:
- Dataset statistics (records, fields, types)
- Category distribution
- Missing value analysis
- Image URL analysis
- Sample products per category

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from collections import Counter

# Style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

## 1. Load Dataset

In [ ]:
DATA_PATH = Path('../data/products.json')

with open(DATA_PATH, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

df = pd.DataFrame(raw_data)
print(f'Total records: {len(df)}')
print(f'Columns: {list(df.columns)}')
print(f'Dtypes:\n{df.dtypes}')
df.head()

## 2. Missing Values & Data Quality

In [ ]:
# Missing values
missing = df.isnull().sum()
print('Missing values per column:')
print(missing)

# Duplicates
dupes = df.duplicated(subset=['product_id']).sum()
print(f'\nDuplicate product_ids: {dupes}')

# Product ID uniqueness
print(f'Unique product_ids: {df["product_id"].nunique()} / {len(df)}')

## 3. Category Distribution

In [ ]:
cat_counts = df['category'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
colors = plt.cm.Set2(np.linspace(0, 1, len(cat_counts)))
cat_counts.plot(kind='barh', ax=axes[0], color=colors)
axes[0].set_title('Products per Category')
axes[0].set_xlabel('Count')
for i, v in enumerate(cat_counts.values):
    axes[0].text(v + 0.5, i, str(v), va='center', fontweight='bold')

# Pie chart
axes[1].pie(cat_counts.values, labels=cat_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90)
axes[1].set_title('Category Distribution')

plt.tight_layout()
plt.savefig('../data/category_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nCategory value counts:\n{cat_counts}')
print(f'\nMost common: {cat_counts.index[0]} ({cat_counts.iloc[0]} products)')
print(f'Least common: {cat_counts.index[-1]} ({cat_counts.iloc[-1]} products)')

## 4. Image Analysis

In [ ]:
# Images per product
df['n_images'] = df['images'].apply(len)

print('Images per product:')
print(df['n_images'].describe())

fig, ax = plt.subplots(figsize=(8, 4))
df['n_images'].value_counts().sort_index().plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Number of Images per Product')
ax.set_xlabel('Image Count')
ax.set_ylabel('Number of Products')
plt.tight_layout()
plt.show()

# Image URL domains
from urllib.parse import urlparse
all_urls = [url for urls in df['images'] for url in urls]
domains = [urlparse(u).netloc for u in all_urls]
print(f'\nTotal image URLs: {len(all_urls)}')
print(f'Unique domains: {Counter(domains)}')

## 5. Sample Products per Category

In [ ]:
for cat in df['category'].unique():
    sample = df[df['category'] == cat].iloc[0]
    print(f'\n--- {cat} ---')
    print(f'  ID:   {sample["product_id"]}')
    print(f'  Name: {sample["name"]}')
    print(f'  Images: {len(sample["images"])}')
    print(f'  First URL: {sample["images"][0][:80]}...')

## 6. Key Findings

| Metric | Value |
|--------|-------|
| Total products | 200 |
| Categories | 7 (Rings, Earrings, Mangalsutras, Chains, Bangles, Pendants, Necklaces) |
| Missing values | 0 |
| Duplicates | 0 |
| Images per product | 2 (consistent) |
| Image source | Shopify CDN |

**Observations:**
- Dataset is clean — no missing values or duplicates
- Class imbalance: Earrings (57) and Rings (56) dominate, Necklaces (9) is underrepresented
- All images hosted on same CDN — consistent quality expected
- Each product has exactly 2 images — we use the first for embedding